In [1]:
import pandas as pd
import matplotlib.pyplot as plt

ab = pd.read_csv("../data/raw/Citycycle_abonnes.csv")
print(ab.shape)
print(ab.isna().sum())
print(ab["formule"].value_counts(dropna=False))
print(ab["a_renouvele"].value_counts(dropna=False))
print(ab.duplicated("abonne_id").sum(), "doublons d'identifiant")

# Annuels uniquement, cible : 1 = n'a pas renouvelé
an = ab[ab["formule"] == "annuel"].copy()
an["churn"] = (an["a_renouvele"] == "non").astype(int)
print(len(an), "abonnés annuels ; taux de churn :", round(an["churn"].mean(), 3))

# 1) Ancienneté
an["anc_bin"] = pd.qcut(an["anciennete_mois"], 5, duplicates="drop")
print(an.groupby("anc_bin", observed=True)["churn"].agg(["mean", "count"]))

# 2) Incidents
print(an.groupby("incidents_signales")["churn"].agg(["mean", "count"]))

# 3) Trajets par mois
an["traj_bin"] = pd.qcut(an["trajets_par_mois"], 5, duplicates="drop")
print(an.groupby("traj_bin", observed=True)["churn"].agg(["mean", "count"]))

# 4) Distance moyenne
an["dist_bin"] = pd.qcut(an["distance_moy_km"], 5, duplicates="drop")
print(an.groupby("dist_bin", observed=True)["churn"].agg(["mean", "count"]))

# 5) Corrélations
print(an[["anciennete_mois", "trajets_par_mois", "distance_moy_km",
          "incidents_signales", "churn"]].corr()["churn"])

(500, 7)
abonne_id             0
anciennete_mois       0
trajets_par_mois      0
distance_moy_km       0
incidents_signales    0
formule               0
a_renouvele           0
dtype: int64
formule
annuel     360
mensuel    140
Name: count, dtype: int64
a_renouvele
oui    385
non    115
Name: count, dtype: int64
0 doublons d'identifiant
360 abonnés annuels ; taux de churn : 0.217
                   mean  count
anc_bin                       
(0.999, 10.0]  0.268293     82
(10.0, 17.0]   0.246575     73
(17.0, 22.0]   0.225806     62
(22.0, 32.0]   0.232877     73
(32.0, 60.0]   0.100000     70
                        mean  count
incidents_signales                 
0                   0.181416    226
1                   0.224299    107
2                   0.500000     24
3                   0.333333      3
                  mean  count
traj_bin                     
(0.999, 4.0]  0.292929     99
(4.0, 6.0]    0.224490     49
(6.0, 9.0]    0.228571     70
(9.0, 14.0]   0.213333     75
(14.

In [2]:
# Seuil d'ancienneté : churn avant / après chaque seuil candidat
for s in range(20, 45, 3):
    av = an[an["anciennete_mois"] <= s]["churn"]
    ap = an[an["anciennete_mois"] > s]["churn"]
    print(f"seuil {s:>2} mois : avant {av.mean():.3f} (n={len(av)}) | après {ap.mean():.3f} (n={len(ap)})")

# Signaux cumulés
an["inc_grp"] = an["incidents_signales"].clip(upper=2)
an["peu_trajets"] = an["trajets_par_mois"] <= 4
print(an.groupby(["inc_grp", "peu_trajets"])["churn"].agg(["mean", "count"]))

# Les variables sont-elles liées entre elles ?
print(an[["anciennete_mois", "trajets_par_mois", "incidents_signales"]].corr())

seuil 20 mois : avant 0.246 (n=195) | après 0.182 (n=165)
seuil 23 mois : avant 0.243 (n=222) | après 0.174 (n=138)
seuil 26 mois : avant 0.233 (n=249) | après 0.180 (n=111)
seuil 29 mois : avant 0.238 (n=273) | après 0.149 (n=87)
seuil 32 mois : avant 0.245 (n=290) | après 0.100 (n=70)
seuil 35 mois : avant 0.236 (n=301) | après 0.119 (n=59)
seuil 38 mois : avant 0.232 (n=311) | après 0.122 (n=49)
seuil 41 mois : avant 0.226 (n=318) | après 0.143 (n=42)
seuil 44 mois : avant 0.219 (n=334) | après 0.192 (n=26)
                         mean  count
inc_grp peu_trajets                 
0       False        0.147929    169
        True         0.280702     57
1       False        0.220779     77
        True         0.233333     30
2       False        0.466667     15
        True         0.500000     12
                    anciennete_mois  trajets_par_mois  incidents_signales
anciennete_mois            1.000000         -0.023050           -0.023128
trajets_par_mois          -0.023050     

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

X = an[["anciennete_mois", "trajets_par_mois", "incidents_signales"]]
y = an["churn"]
cv = StratifiedKFold(5, shuffle=True, random_state=42)
auc = cross_val_score(LogisticRegression(class_weight="balanced", max_iter=1000),
                      X, y, cv=cv, scoring="roc_auc")
print("AUC par pli :", auc.round(3), "| moyenne :", auc.mean().round(3))

AUC par pli : [0.664 0.714 0.614 0.483 0.655] | moyenne : 0.626


In [4]:
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Estimation plus stable : 5 plis x 10 répétitions
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

modeles = {
    "logistique": make_pipeline(StandardScaler(),
                                LogisticRegression(class_weight="balanced", max_iter=1000)),
    "arbre prof. 2": DecisionTreeClassifier(max_depth=2, min_samples_leaf=20,
                                            class_weight="balanced", random_state=42),
    "arbre prof. 3": DecisionTreeClassifier(max_depth=3, min_samples_leaf=20,
                                            class_weight="balanced", random_state=42),
}
for nom, m in modeles.items():
    s = cross_val_score(m, X, y, cv=cv, scoring="roc_auc")
    print(f"{nom:<15} AUC moyen {s.mean():.3f} | écart-type {s.std():.3f}")

# Coefficients de la logistique (variables standardisées : comparables entre elles)
m = modeles["logistique"].fit(X, y)
for nom, c in zip(X.columns, m[-1].coef_[0]):
    print(f"{nom:<20} {c:+.3f}")

logistique      AUC moyen 0.638 | écart-type 0.070
arbre prof. 2   AUC moyen 0.550 | écart-type 0.056
arbre prof. 3   AUC moyen 0.575 | écart-type 0.068
anciennete_mois      -0.235
trajets_par_mois     -0.450
incidents_signales   +0.315
